# Day 10 – Feature Engineering + SageMaker Feature Store

This notebook transforms raw customer data into reusable ML features. It does not create AWS resources.

## 1. Load raw data

In [ ]:
import pandas as pd

df = pd.read_csv("customer_data.csv")
df.head()


## 2. Convert date columns

In [ ]:
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.dtypes


## 3. Create features

In [ ]:
# Fixed reference date for reproducible calculations
today = pd.Timestamp("2026-10-30")

df["age"] = (
    today.year - df["date_of_birth"].dt.year
    - (
        (today.month < df["date_of_birth"].dt.month)
        | ((today.month == df["date_of_birth"].dt.month)
           & (today.day < df["date_of_birth"].dt.day))
    )
)

df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.month

df.head()


## 4. Select reusable feature columns

In [ ]:
feature_df = df[
    ["customer_id", "age", "hour", "weekday", "month", "purchase_amount"]
].copy()
feature_df


## 5. Save engineered features

In [ ]:
feature_df.to_csv("engineered_features.csv", index=False)
print("Feature engineering completed!")
print("Dataset shape:", feature_df.shape)


## 6. Check missing values

In [ ]:
feature_df.isnull().sum()


## 7. Map to Feature Store concepts

In [ ]:
feature_store_mapping = {
    "Feature Group": "CustomerFeatures (conceptual example)",
    "Record Identifier": "customer_id",
    "Features": ["age", "hour", "weekday", "month", "purchase_amount"],
    "Event Time": "timestamp (source timestamp)"
}
feature_store_mapping


## 8. Online Store vs Offline Store

- **Online Store:** low-latency retrieval for real-time inference.
- **Offline Store:** historical feature data for training and analysis, persisted in Amazon S3.

This notebook only prepares feature data; it does not create a SageMaker Feature Group or chargeable AWS resources.